# CSE 151B Competition — Google Colab Submission Notebook

This notebook runs the full inference pipeline on Google Colab (no vLLM).

**Steps:**
1. Install dependencies
2. Upload `private.jsonl`
3. Load model (bf16 on A100, or 4-bit on T4/L4)
4. Run two-stage MCQ + free-form inference
5. Save and download the submission CSV

> **Runtime**: Runtime → Change runtime type → **A100 GPU** (Colab Pro).

## 1. Install Dependencies

Run once. After the cell finishes, **do NOT restart** — the kernel is already using the newly installed packages.

In [ ]:
# Run once — installs take ~2 minutes
!pip install -q transformers accelerate bitsandbytes sympy antlr4-python3-runtime==4.11.1 tqdm
print('Done.')

## 2. Upload Files

Upload `private.jsonl` and `judger.py` from your local machine.

In [ ]:
from google.colab import files
import os

print('Upload private.jsonl, judger.py, and utils.py when prompted.')
uploaded = files.upload()
for fname in uploaded:
    print(f'  uploaded: {fname} ({len(uploaded[fname])} bytes)')

os.makedirs('data', exist_ok=True)
os.makedirs('results/cache', exist_ok=True)

if 'private.jsonl' in uploaded:
    with open('data/private.jsonl', 'wb') as f:
        f.write(uploaded['private.jsonl'])
    print('Saved -> data/private.jsonl')

for fname in ('judger.py', 'utils.py'):
    if fname in uploaded:
        with open(fname, 'wb') as f:
            f.write(uploaded[fname])
        print(f'Saved -> {fname}')

## 3. Imports & Configuration

In [ ]:
import json
import os
import re
import sys
import csv
import hashlib
from pathlib import Path
from typing import Optional
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = 'Qwen/Qwen3-4B-Thinking-2507'
DATA_PATH   = 'data/private.jsonl'
OUTPUT_PATH = 'results/submission_colab.csv'
MAX_NEW_TOKENS = 32768
SEED        = 13

# Detect GPU type to choose quantization
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}  |  VRAM: {total_vram:.1f} GB')
    # A100 has 40 GB — run bf16. T4 has 16 GB, L4 has 24 GB — use 4-bit.
    USE_4BIT = total_vram < 35.0
else:
    print('WARNING: No GPU found!')
    USE_4BIT = True

print(f'USE_4BIT quantization: {USE_4BIT}')

## 4. Load Dataset

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get('options')) for d in data)
n_free = sum(not d.get('options')   for d in data)
print(f'Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)')

## 5. Preprocessing (LaTeX Repair)

In [ ]:
LATEX_CMDS = [
    'int', 'iint', 'iiint', 'oint', 'sum', 'prod', 'lim',
    'infty', 'partial',
    'frac', 'dfrac', 'tfrac', 'sqrt', 'binom',
    'sin', 'cos', 'tan', 'cot', 'sec', 'csc',
    'arcsin', 'arccos', 'arctan',
    'sinh', 'cosh', 'tanh',
    'log', 'ln', 'exp',
    'alpha', 'beta', 'gamma', 'delta', 'epsilon', 'zeta', 'eta',
    'theta', 'iota', 'kappa', 'lambda', 'mu', 'nu',
    'xi', 'pi', 'rho', 'sigma', 'tau', 'phi', 'chi', 'psi', 'omega',
    'Gamma', 'Delta', 'Theta', 'Lambda', 'Pi', 'Sigma', 'Phi', 'Psi', 'Omega',
    'pm', 'mp', 'times', 'cdot', 'leq', 'geq', 'neq',
    'approx', 'equiv', 'sim', 'to', 'in', 'notin',
    'subset', 'supset', 'cup', 'cap',
    'mathbb', 'mathrm', 'mathbf', 'mathcal',
    'left', 'right', 'text',
]

_LATEX_MATH_CONTEXT = r'[{}_^()]'
_LATEX_PATTERNS = [
    (re.compile(rf'(?<!\\)\b{cmd}(?={_LATEX_MATH_CONTEXT})'), rf'\\{cmd}')
    for cmd in LATEX_CMDS
]

def repair_latex(text: str) -> str:
    for pattern, repl in _LATEX_PATTERNS:
        text = pattern.sub(repl, text)
    return text

def preprocess_question(item: dict) -> dict:
    out = dict(item)
    out['question'] = repair_latex(item['question'])
    if item.get('options'):
        out['options'] = [repair_latex(opt) for opt in item['options']]
    return out

print('LaTeX repair functions defined.')

## 6. Prompt Construction

In [ ]:
SYSTEM_PROMPT_MATH = (
    'Please reason step by step, and put your final answer within \\boxed{}. '
    'If the problem has multiple sub-answers, separate them by commas inside a '
    'single \\boxed{}, e.g. \\boxed{3, 7}.'
)

SYSTEM_PROMPT_MCQ = (
    'Please reason step by step, then select the single best option from the answer '
    'choices. The question may contain transcription errors (e.g. missing backslashes); '
    'interpret it charitably. Output ONLY the letter of your chosen option inside '
    '\\boxed{}, e.g. \\boxed{C}.'
)

MCQ_STAGE2_INSTRUCTION = (
    'Your previous answer does not match any of the options shown above. The '
    'question may contain transcription errors (e.g. missing backslashes); '
    'reconsider charitably and choose the option that would make the problem '
    'mathematically sensible. Output ONLY the letter inside \\boxed{}, e.g. \\boxed{C}.'
)

SYSTEM_PROMPT_LONG = (
    'Before solving, restate the problem in your own words and list every given '
    'condition and the quantity to find. Then solve.\n\n'
    'Please reason step by step, and put your final answer within \\boxed{}. '
    'If the problem has multiple sub-answers, separate them by commas inside a '
    'single \\boxed{}, e.g. \\boxed{3, 7}.'
)

SYSTEM_PROMPT_MULTIPART = (
    'This problem contains multiple sub-answers marked by [ANS] placeholders '
    'in the question. Identify each sub-question, then solve them in the order '
    'they appear.\n\n'
    'After completing all reasoning, output every answer in a SINGLE \\boxed{}, '
    'comma-separated, in the same order as the [ANS] slots — for example '
    '\\boxed{a, b, c}. Do not split answers across multiple \\boxed{} expressions.'
)

_OPTIONS_REFERENCED = re.compile(
    r'\b(?:'
    r'which (?:of the )?(?:following|options?|statements?|choices?)'
    r'|(?:all|none) of the above'
    r'|select all'
    r'|what (?:is|are) (?:correct|right|true)'
    r'|determine the (?:corresponding )?output'
    r')\b',
    re.IGNORECASE,
)
_OPT_SELF_REFERENCE = re.compile(r'\b(?:all|none) of the above\b', re.IGNORECASE)

def mcq_needs_options(question: str, options: list) -> bool:
    if _OPTIONS_REFERENCED.search(question):
        return True
    return any(_OPT_SELF_REFERENCE.search(opt) for opt in options)

def select_prompt(item: dict) -> tuple:
    question = item['question']
    options  = item.get('options')
    if not options:
        n_ans = question.count('[ANS]')
        qlen  = len(question)
        if n_ans > 1:
            system = SYSTEM_PROMPT_MULTIPART
        elif qlen > 500:
            system = SYSTEM_PROMPT_LONG
        else:
            system = SYSTEM_PROMPT_MATH
        return system, question, False
    if mcq_needs_options(question, options):
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = '\n'.join(f'{lbl}. {opt.strip()}' for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f'{question}\n\nOptions:\n{opts_text}', True
    return SYSTEM_PROMPT_MATH, question, False

def build_mcq_stage2_messages(item: dict, stage1_response: str) -> list:
    question  = item['question']
    options   = item['options']
    labels    = [chr(65 + i) for i in range(len(options))]
    opts_text = '\n'.join(f'{lbl}. {opt.strip()}' for lbl, opt in zip(labels, options))
    return [
        {'role': 'system',    'content': SYSTEM_PROMPT_MATH},
        {'role': 'user',      'content': question},
        {'role': 'assistant', 'content': stage1_response},
        {'role': 'user',      'content': f'Options:\n{opts_text}\n\n{MCQ_STAGE2_INSTRUCTION}'},
    ]

print('Prompt helpers defined.')

## 7. Load Model

- **A100 (≥35 GB VRAM)**: loads in `bfloat16` — fastest, best quality.
- **L4 / T4 (<35 GB)**: loads in 4-bit NF4 — fits in 16 GB VRAM.

Downloading Qwen3-4B takes ~5 minutes on the first run; subsequent runs use the Colab cache.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
    )

model.eval()
print(f'Model loaded on {next(model.parameters()).device}.')

## 8. Generate Responses

Runs sequentially (one question at a time). The cache is written after every question so you can resume safely after a crash or timeout.

Estimated time: ~30–60 s/question on A100 → ~10–18 h for all 1126 questions at `MAX_NEW_TOKENS=32768`. Set `MAX_NEW_TOKENS=8192` to go ~4× faster with minimal accuracy loss.

In [ ]:
PIPELINE_TAG = 'v2.6'
prompt_hash = hashlib.md5(
    (SYSTEM_PROMPT_MATH + '||' + SYSTEM_PROMPT_MCQ + '||'
     + MCQ_STAGE2_INSTRUCTION + '||' + SYSTEM_PROMPT_LONG + '||'
     + SYSTEM_PROMPT_MULTIPART + '||' + PIPELINE_TAG).encode()
).hexdigest()[:8]
CACHE_PATH = f'results/cache/colab_{prompt_hash}_seed{SEED}.jsonl'
Path(CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)

cache = {}
if Path(CACHE_PATH).exists():
    with open(CACHE_PATH) as f:
        for line in f:
            e = json.loads(line)
            cache[e['id']] = e['response']

to_generate = [item for item in data if item['id'] not in cache]
print(f'Cache hits: {len(cache)} / {len(data)}')
print(f'To generate: {len(to_generate)}')

In [ ]:
def verify_against_options(response: str, options: list) -> Optional[str]:
    sys.path.insert(0, '.')
    from judger import Judger
    judger_local = Judger(strict_extract=True)
    for i, opt in enumerate(options):
        try:
            if judger_local.auto_judge(pred=response, gold=[opt], options=[[]]):
                return chr(65 + i)
        except Exception:
            continue
    return None

def generate_response(messages: list) -> str:
    """Run one forward pass and return the generated text."""
    prompt_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(prompt_str, return_tensors='pt').to(model.device)
    input_ids      = encoded['input_ids']
    attention_mask = encoded['attention_mask']

    with torch.no_grad():
        out = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('Helpers defined.')

In [ ]:
cache_file = open(CACHE_PATH, 'a')

try:
    for item in tqdm(to_generate, desc='Generating'):
        prepped = preprocess_question(item)
        sys_p, user_p, opts_visible = select_prompt(prepped)

        # Stage 1
        stage1_msgs = [
            {'role': 'system', 'content': sys_p},
            {'role': 'user',   'content': user_p},
        ]
        stage1_text = generate_response(stage1_msgs)

        final_response = stage1_text
        stage1_field   = None

        if item.get('options') and not opts_visible:
            # Hidden-options MCQ: try to match stage-1 answer
            stage1_field = stage1_text
            letter = verify_against_options(stage1_text, prepped['options'])
            if letter is not None:
                final_response = f'\\boxed{{{letter}}}'
            else:
                # Stage 2: show options and reconcile
                s2_msgs = build_mcq_stage2_messages(prepped, stage1_text)
                final_response = generate_response(s2_msgs)

        # Save to cache immediately
        rec = {'id': item['id'], 'response': final_response}
        if stage1_field is not None:
            rec['stage1'] = stage1_field
        cache[item['id']] = final_response
        cache_file.write(json.dumps(rec) + '\n')
        cache_file.flush()

finally:
    cache_file.close()

print(f'Generation complete. Total cached: {len(cache)}')

## 9. Save Submission CSV

In [ ]:
responses = [cache[item['id']] for item in data]

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w', newline='') as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(['id', 'response'])
    for item, resp in zip(data, responses):
        writer.writerow([item['id'], resp])

print(f'Saved {len(data)} rows to {out_path}')

## 10. Download Results

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)
print('Download started.')

## 11. Score & Summary\n\nIf you are running on `public.jsonl` (which has gold answers), this section scores every response and prints a full accuracy breakdown.\n\nIf you are running on `private.jsonl` (no gold answers), it skips scoring and just prints response statistics."

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r'\\boxed\{([A-Za-z])\}', text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ''

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

def acc(subset):
    return sum(r['correct'] for r in subset) / len(subset) * 100 if subset else 0.0

# ── Check if gold answers are present ─────────────────────────────────────────
HAS_GOLD = 'answer' in data[0]

if not HAS_GOLD:
    # Private set — no scoring, just print stats
    print('Running on private set — no gold answers available.')
    print()
    n_mcq_resp  = sum(1 for item in data if item.get('options'))
    n_free_resp = sum(1 for item in data if not item.get('options'))
    boxed_mcq   = sum(
        1 for item, resp in zip(data, responses)
        if item.get('options') and re.search(r'\\boxed\{[A-Z]\}', resp)
    )
    print(f'Total questions : {len(data)}')
    print(f'  MCQ           : {n_mcq_resp}  ({boxed_mcq} have a \\boxed{{LETTER}})')
    print(f'  Free-form     : {n_free_resp}')
    avg_len = sum(len(r) for r in responses) / len(responses)
    print(f'Avg response len: {avg_len:.0f} chars')

else:
    # Public set — full scoring
    sys.path.insert(0, '.')
    from judger import Judger
    judger = Judger(strict_extract=False)

    results = []
    for item, response in tqdm(zip(data, responses), total=len(data), desc='Scoring'):
        is_mcq = bool(item.get('options'))
        gold   = item['answer']
        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
            except Exception:
                correct = False
        results.append({'id': item['id'], 'is_mcq': is_mcq, 'gold': gold,
                        'response': response, 'correct': correct})

    mcq_res  = [r for r in results if r['is_mcq']]
    free_res = [r for r in results if not r['is_mcq']]

    print('=' * 50)
    print('EVALUATION RESULTS')
    print('=' * 50)
    print(f'  MCQ        : {sum(r["correct"] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)')
    print(f'  Free-form  : {sum(r["correct"] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)')
    print(f'  Overall    : {sum(r["correct"] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)')
    print('=' * 50)

    # By question length
    sorted_q = sorted(data, key=lambda d: len(d['question']))
    bsize = len(sorted_q) // 3
    length_buckets = {
        'short ': [d['id'] for d in sorted_q[:bsize]],
        'medium': [d['id'] for d in sorted_q[bsize:2*bsize]],
        'long  ': [d['id'] for d in sorted_q[2*bsize:]],
    }
    print('\nBy question length:')
    for label, ids in length_buckets.items():
        s = [r for r in results if r['id'] in set(ids)]
        if s:
            print(f'  {label}: {sum(r["correct"] for r in s):3d} / {len(s):3d}  ({acc(s):.2f}%)')

    # By routing path (MCQ only)
    print('\nBy routing path (MCQ only):')
    for label, pred in [
        ('visible-opts', lambda it: mcq_needs_options(it['question'], it['options'])),
        ('hidden-opts ', lambda it: not mcq_needs_options(it['question'], it['options'])),
    ]:
        ids = {it['id'] for it in data if it.get('options') and pred(it)}
        s = [r for r in results if r['id'] in ids]
        if s:
            print(f'  {label}: {sum(r["correct"] for r in s):3d} / {len(s):3d}  ({acc(s):.2f}%)')